# Full Demo: Energy Commodities Systematic Trading Platform

End-to-end demonstration of the complete trading system:

1. **Build Forward Curves** from EIA/synthetic settlement data
2. **Seasonal Decomposition** and convenience yield extraction
3. **Risk Analytics** -- parallel delta, scenario P&L
4. **Carry & Roll Yield** analysis across the term structure
5. **Alpha Signal Generation** -- composite multi-factor model
6. **RFQ Generation & Quote Optimization** with win-probability model
7. **Execution Planning** -- Almgren-Chriss market impact
8. **KDB+ Storage & Databento L2 Data**

In [ ]:
import sys
import time
from pathlib import Path
from datetime import date

project_root = Path.cwd()
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import ENERGY_PRODUCTS
from shared.plot_style import apply_style, COLOURS
apply_style()

print('Project root:', project_root)
print('Products:', list(ENERGY_PRODUCTS.keys()))

---
## Step 1: Data Loading & Forward Curve Construction

In [ ]:
from module_a_curves.data_loader import CommodityDataLoader
from module_a_curves.curve_bootstrapper import (
    ForwardCurve, ForwardCurveBootstrapper, FuturesSettlement,
)
from module_a_curves.seasonal_model import SeasonalForwardCurve

loader = CommodityDataLoader()
bootstrapper = ForwardCurveBootstrapper()
curves = {}

for product in ["CL", "HO", "RB", "NG"]:
    strip = loader.get_strip_for_date(product=product)
    if strip and len(strip) > 0:
        settlements = [
            FuturesSettlement(
                product=s["product"], contract_code=s["contract_code"],
                settlement=s["settlement"], time_to_expiry=s["time_to_expiry"],
            ) for s in strip
        ]
        curve = bootstrapper.bootstrap(settlements, product=product)
    else:
        times = [(i + 1) / 12 for i in range(12)]
        _synth = {
            "CL": (72.50, [0.00, 0.80, 1.30, 1.55, 1.65, 1.68, 1.60, 1.45, 1.20, 0.85, 0.40, -0.10]),
            "HO": (2.35,  [0.00, 0.04, 0.06, 0.05, 0.02, -0.02, -0.06, -0.10, -0.13, -0.15, -0.16, -0.15]),
            "RB": (2.45,  [0.00, 0.06, 0.10, 0.11, 0.08, 0.03, -0.04, -0.10, -0.15, -0.18, -0.19, -0.18]),
            "NG": (3.80,  [0.00, -0.12, -0.20, -0.22, -0.15, -0.03, 0.15, 0.38, 0.55, 0.60, 0.48, 0.30]),
        }
        base, offsets = _synth.get(product, (72.50, [0.10 * i for i in range(12)]))
        prices = [base + o for o in offsets]
        curve = ForwardCurve(times, prices, product=product)
    curves[product] = curve
    print(f"{product}: {len(curve.times)} tenors, front={curve.forward_price(curve.times[0]):.2f}, "
          f"contango={'Yes' if curve.is_contango() else 'No'}")

cl_curve = curves["CL"]

print(f"\nConvenience Yield (CL):")
for t in [0.25, 0.5, 1.0]:
    cy = cl_curve.convenience_yield(t)
    print(f"  CY at {t:.2f}y: {cy*100:.2f}%")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
_units = {"CL": "$/bbl", "HO": "$/gal", "RB": "$/gal", "NG": "$/MMBtu"}
from config import ENERGY_PRODUCTS as _EP
for ax, (product, curve) in zip(axes.flat, curves.items()):
    tenors = curve.times
    prices = [curve.forward_price(t) for t in tenors]
    ax.plot(tenors, prices, 'o-', color=COLOURS[product], linewidth=2, markersize=5)
    ax.fill_between(tenors, prices, min(prices) - (max(prices) - min(prices)) * 0.1,
                     alpha=0.15, color=COLOURS[product])
    pmin, pmax = min(prices), max(prices)
    margin = max((pmax - pmin) * 0.3, pmax * 0.005)
    ax.set_ylim(pmin - margin, pmax + margin)
    ax.set_xlabel("Tenor (years)")
    ax.set_ylabel(f"Price ({_units.get(product, '$/unit')})")
    shape = "contango" if curve.is_contango() else "backwardation"
    ax.set_title(f"{_EP[product]} ({product}) — {shape}")
    ax.grid(True, alpha=0.3)
fig.suptitle("Energy Forward Curves", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Step 2: Risk Analytics & Scenario Analysis

In [ ]:
from module_b_trading.futures_pricer import FuturesPricer, FuturesPosition
from module_b_trading.risk_analytics import RiskAnalytics
from module_b_trading.scenario_engine import ScenarioEngine, STANDARD_SCENARIOS

cl_strip = loader.get_strip_for_date(product="CL") or []
if cl_strip:
    cl_settlements = [
        FuturesSettlement(
            product=s["product"], contract_code=s["contract_code"],
            settlement=s["settlement"], time_to_expiry=s["time_to_expiry"],
        ) for s in cl_strip
    ]
else:
    cl_settlements = [
        FuturesSettlement(
            product="CL", contract_code=f"CL{i+1:02d}",
            settlement=cl_curve.forward_price(t), time_to_expiry=t,
        ) for i, t in enumerate(cl_curve.times)
    ]

pricer = FuturesPricer(cl_curve)
risk = RiskAnalytics(bootstrapper, cl_settlements)

portfolio = [
    FuturesPosition("CLK26", "CL", 50, "long", 72.50, "2026-04-20"),
    FuturesPosition("CLN26", "CL", 30, "long", 73.20, "2026-06-22"),
    FuturesPosition("CLZ26", "CL", 20, "short", 74.50, "2026-11-20"),
]

print("Portfolio:")
for pos in portfolio:
    print(f"  {pos}")

mtm_df = pricer.portfolio_mtm(portfolio)
print(f"\n{mtm_df.to_string(index=False)}")
print(f"\nTotal MTM: ${mtm_df['mtm_usd'].sum():,.0f}")

print("\nParallel Delta:")
total_delta = 0.0
for pos in portfolio:
    delta = risk.parallel_delta(pos)
    total_delta += delta
    print(f"  {pos.ticker}: ${delta:,.0f}")
print(f"  Net: ${total_delta:,.0f}")

In [ ]:
scenario_engine = ScenarioEngine(bootstrapper, cl_settlements)

print("Scenario P&L:")
print(f"{'Scenario':<35} {'P&L ($)':>12}")
print("-" * 50)
for name, scenario in list(STANDARD_SCENARIOS.items())[:6]:
    try:
        results = scenario_engine.run_scenario(scenario, portfolio)
        total = results["scenario_pnl"].sum() if "scenario_pnl" in results.columns else 0
        print(f"  {name:<33} ${total:>12,.0f}")
    except Exception as e:
        print(f"  {name:<33} Error: {e}")

---
## Step 3: Roll Yield & Carry Analysis

In [ ]:
from module_b_trading.carry_rolldown import RollYieldCalculator

roll_calc = RollYieldCalculator(cl_curve)
matrix = roll_calc.roll_yield_matrix("CL")

if len(matrix) > 0:
    print(matrix[["front_tenor", "back_tenor", "roll_yield_ann", "regime"]].to_string(index=False))
    
    best = roll_calc.best_roll_trades("CL", top_n=3)
    if (best["roll_yield_ann"] < 0).all():
        label = "Largest carry cost (contango)"
    elif (best["roll_yield_ann"] > 0).all():
        label = "Best carry trades (backwardation)"
    else:
        label = "Top roll-yield trades"
    print(f"\n{label}:")
    for _, row in best.iterrows():
        print(f"  {row['front_tenor']:.2f}y -> {row['back_tenor']:.2f}y: "
              f"{row['roll_yield_ann']*100:+.1f}% ({row['regime']})")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
if len(matrix) > 0:
    yields = matrix["roll_yield_ann"].values * 100
    labels = [f"{r['front_tenor']:.1f}-{r['back_tenor']:.1f}" for _, r in matrix.iterrows()]
    colors = [COLOURS["accent"] if y > 0 else COLOURS["warning"] for y in yields]
    ax.bar(range(len(yields)), yields, color=colors, alpha=0.8)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_xlabel("Tenor Pair (years)")
    ax.set_ylabel("Annualized Roll Yield (%)")
    ax.set_title("CL Roll Yield by Tenor Pair")
    ax.axhline(y=0, color="black", linewidth=0.8)
    ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

---
## Step 4: Alpha Signal Generation

In [ ]:
from module_b_trading.alpha_signals import CompositeAlphaModel

alpha_model = CompositeAlphaModel.default_crude_model()
front_price = cl_curve.forward_price(cl_curve.times[0])
deferred_price = cl_curve.forward_price(cl_curve.times[-1])

alpha_result = alpha_model.compute_composite(
    forward_curve=cl_curve,
    front_price=front_price,
    deferred_price=deferred_price,
    inventory_level=440.0,
    month=3,
    price=front_price,
    cl_price=front_price,
    ho_price=2.35,
    rb_price=2.45,
)

print("Signal breakdown:")
for name, val in alpha_result.items():
    bar = "+" * max(0, int(val * 5)) + "-" * max(0, int(-val * 5))
    print(f"  {name:20s}: {val:+.3f}  {bar}")

composite = alpha_result["composite"]
if composite > 0.5:
    print(f"\n>> BULLISH bias ({composite:+.2f})")
elif composite < -0.5:
    print(f"\n>> BEARISH bias ({composite:+.2f})")
else:
    print(f"\n>> NEUTRAL ({composite:+.2f})")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
signal_names = list(alpha_result.keys())
signal_vals = list(alpha_result.values())
colors = [COLOURS["accent"] if v > 0 else COLOURS["warning"] for v in signal_vals]
ax.barh(signal_names, signal_vals, color=colors, alpha=0.8)
ax.set_xlabel("Signal Value")
ax.set_title("Composite Alpha Signal Breakdown (CL)")
ax.axvline(x=0, color="black", linewidth=0.8)
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

---
## Step 5: RFQ Generation & Quote Optimization

In [ ]:
from module_b_trading.rfq_generator import RFQGenerator
from module_b_trading.win_probability import WinProbabilityModel
from module_b_trading.quote_optimizer import QuoteOptimizer

rfq_gen = RFQGenerator(seed=42, rfqs_per_day=50)
rfqs = rfq_gen.generate_batch(n=50)
print(f"Generated {len(rfqs)} RFQs")
print(f"Product mix: {rfqs['product'].value_counts().to_dict()}")

win_model = WinProbabilityModel()
optimizer = QuoteOptimizer(win_model=win_model)
mid_prices = {"CL": 72.50, "HO": 2.35, "RB": 2.45, "NG": 3.80}
quotes = optimizer.optimize_batch(rfqs, mid_prices)

print(f"\nOptimised {len(quotes)} quotes")
print(f"Avg win probability: {quotes['win_probability'].mean():.1%}")
print(f"Total E[PnL]: ${quotes['expected_pnl'].sum():,.0f}")
print(f"\nBy product:")
for product in ["CL", "HO", "RB", "NG"]:
    mask = quotes["product"] == product
    if mask.any():
        sub = quotes[mask]
        print(f"  {product}: {len(sub)} quotes, avg spread={sub['spread'].mean():.4f}, "
              f"E[PnL]=${sub['expected_pnl'].sum():,.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(quotes["win_probability"], bins=20, color=COLOURS["primary"], alpha=0.7, edgecolor="white")
axes[0].set_xlabel("Win Probability")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Win Probability Distribution (avg={quotes['win_probability'].mean():.1%})")
axes[0].grid(True, alpha=0.3, axis="y")

# E[PnL] by product
for product in ["CL", "HO", "RB", "NG"]:
    mask = quotes["product"] == product
    if mask.any():
        axes[1].hist(quotes.loc[mask, "expected_pnl"], bins=15, alpha=0.6, label=product, color=COLOURS[product])
axes[1].set_xlabel("Expected P&L ($)")
axes[1].set_ylabel("Count")
axes[1].set_title("E[PnL] Distribution by Product")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

---
## Step 6: Execution Planning & Market Impact

In [ ]:
from module_c_execution.market_impact import AlmgrenChrissModel
from module_c_execution.execution_scheduler import TWAPScheduler, VWAPScheduler, AdaptiveScheduler
from module_c_execution.order_simulator import OrderSimulator

impact_model = AlmgrenChrissModel()
print("Almgren-Chriss Impact Estimates:")
for product, size in [("CL", 100), ("HO", 50), ("RB", 50), ("NG", 75)]:
    est = impact_model.estimate_impact(product, size)
    print(f"  {product} {size} lots: cost=${est.total_cost_usd:,.0f} ({est.cost_bps:.1f}bps)")

print("\nStrategy Comparison (CL 200 lots):")
comparison = impact_model.compare_strategies("CL", 200)
print(comparison.to_string(index=False))

trajectory = impact_model.optimal_trajectory("CL", 200, n_slices=10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(range(len(trajectory)), trajectory, 'o-', color=COLOURS["primary"], linewidth=2)
axes[0].fill_between(range(len(trajectory)), trajectory, alpha=0.2, color=COLOURS["primary"])
axes[0].set_xlabel("Time Slice")
axes[0].set_ylabel("Cumulative Fraction Executed")
axes[0].set_title("Almgren-Chriss Optimal Trajectory (CL 200 lots)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(comparison["horizon_min"], comparison["cost_bps"], 's-', color=COLOURS["secondary"], linewidth=2)
axes[1].set_xlabel("Execution Horizon (minutes)")
axes[1].set_ylabel("Total Cost (bps)")
axes[1].set_title("Market Impact vs Execution Horizon")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 7: KDB+ Storage & Databento L2 Data

In [ ]:
from shared.kdb_interface import KDBInterface, KDBConfig
from config import KDB_HOST, KDB_PORT

try:
    kdb = KDBInterface(KDBConfig(host=KDB_HOST, port=KDB_PORT))
    kdb.create_tables()
    
    fwd_rows = []
    for t in cl_curve.times:
        fwd_rows.append({
            "dt": "2026-03-10", "product": "CL", "tenor": t,
            "price": cl_curve.forward_price(t),
        })
    kdb.insert_forwards(pd.DataFrame(fwd_rows))
    print(f"Stored {len(fwd_rows)} forward curve points")
    
    kdb.insert_inventory(pd.DataFrame([{
        "dt": "2026-03-10", "series": "crude_stocks",
        "val": 440.0, "unt": "MMbbl",
    }]))
    print(f"Stored inventory snapshot")
    print(f"Table counts: {kdb.table_counts()}")
    kdb.close()
except Exception as e:
    print(f"KDB+ not available: {e}")
    print("(Start KDB+ on localhost:5000 to enable storage)")

---
## Summary

| Module | Component | Key Feature |
|--------|-----------|-------------|
| **A** | Data Loader | EIA API for futures strips + inventory, synthetic fallback |
| **A** | Forward Curve | Bootstrap from settlements, log-linear & monotone convex interpolation |
| **A** | Seasonal Model | Fourier decomposition of seasonal price patterns |
| **B** | Futures Pricer | MTM, P&L, contract spec lookup |
| **B** | Risk Analytics | Bump-and-revalue parallel delta, cross-product scenarios |
| **B** | Scenario Engine | Parallel shifts, curve twists, spread compression |
| **B** | Roll Yield | Annualized carry/rolldown matrix, contango/backwardation regime |
| **B** | Alpha Signals | Composite model: term structure, inventory, seasonal, momentum, crack |
| **B** | RFQ Generator | Synthetic RFQ flow with client segments and urgency tiers |
| **B** | Win Probability | Logistic regression P(win | spread, product, urgency) |
| **B** | Quote Optimizer | E[PnL]-maximizing grid search across candidate spreads |
| **B** | Markout P&L | Post-trade decomposition: edge, carry, curve move |
| **C** | Market Impact | Almgren-Chriss model with NYMEX energy calibration |
| **C** | Execution Scheduler | TWAP, VWAP, Adaptive strategies |
| **C** | Order Simulator | Synthetic order book + fill simulation |
| **Shared** | KDB+ Interface | q-native table storage with auto-start |
| **Shared** | Databento Loader | CME Globex L2 book snapshots + trades |
| **Shared** | C++ Kernel | Cubic spline interpolation via pybind11 |

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel 1: CL forward curve
cl_tenors = cl_curve.times
cl_prices = [cl_curve.forward_price(t) for t in cl_tenors]
axes[0,0].plot(cl_tenors, cl_prices, 'o-', color=COLOURS["CL"], linewidth=2)
axes[0,0].set_xlabel("Tenor (years)")
axes[0,0].set_ylabel("Price ($/bbl)")
axes[0,0].set_title("WTI Crude Oil Forward Curve")
axes[0,0].grid(True, alpha=0.3)

# Panel 2: Alpha signals
signal_names = list(alpha_result.keys())
signal_vals = list(alpha_result.values())
bar_colors = [COLOURS["accent"] if v > 0 else COLOURS["warning"] for v in signal_vals]
axes[0,1].barh(signal_names, signal_vals, color=bar_colors, alpha=0.8)
axes[0,1].set_xlabel("Signal Value")
axes[0,1].set_title("Alpha Signal Breakdown")
axes[0,1].axvline(x=0, color="black", linewidth=0.8)
axes[0,1].grid(True, alpha=0.3, axis="x")

# Panel 3: Win probability
axes[1,0].hist(quotes["win_probability"], bins=20, color=COLOURS["primary"], alpha=0.7, edgecolor="white")
axes[1,0].set_xlabel("Win Probability")
axes[1,0].set_ylabel("Count")
axes[1,0].set_title(f"Win Prob Distribution (avg={quotes['win_probability'].mean():.1%})")
axes[1,0].grid(True, alpha=0.3, axis="y")

# Panel 4: Execution trajectory
axes[1,1].plot(range(len(trajectory)), trajectory, 'o-', color=COLOURS["secondary"], linewidth=2)
axes[1,1].fill_between(range(len(trajectory)), trajectory, alpha=0.2, color=COLOURS["secondary"])
axes[1,1].set_xlabel("Time Slice")
axes[1,1].set_ylabel("Cumulative Fraction")
axes[1,1].set_title("Optimal Execution Trajectory (CL)")
axes[1,1].grid(True, alpha=0.3)

fig.suptitle("Energy Commodities Systematic Trading Platform", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()